# A2 · Exploring data with DuckDB and JupySQL

**The question:** the operations team believes that *some shipping modes deliver much later
than others* (trucks? mail?). Is that true?

You will explore `lakehouse.samples.lineitem` (one row per item of an order, with the dates
it was promised and received), mostly with **DuckDB**, check your numbers with **Trino**, draw
a chart, and publish the answer as a table others can use.

Work top to bottom with `Shift+Enter`. The lesson (`README.md` in this folder) explains every
step in more detail.

## 0. Connect to two engines

- **DuckDB** runs *inside your workspace*, on your share of CPU and memory. It reads the
  lakehouse's files directly, with short-lived credentials the catalog gives *you*.
- **Trino** is the lab's *shared* SQL engine (the one you used in A1).

Both see the same tables, as you. The first line of every `%%sql` cell names the engine:
`%%sql duck` or `%%sql trino`.

In [ ]:
import json
import time

import lakehouse
from IPython.display import display

%load_ext sql
%config SqlMagic.displaylimit = 20

duck = lakehouse.duckdb_connect()     # DuckDB, with the lab catalog attached as "lakehouse"
trino = lakehouse.sql_engine()        # Trino, as you
%sql duck --alias duck
%sql trino --alias trino

me = lakehouse.whoami()
schema = f"dbt_{me}"
print(f"You are {me}. Your schema is lakehouse.{schema}")

Expected: `You are <your user name>. Your schema is lakehouse.dbt_<your user name>`.

## 1. How big is the table?

Always know the size before you run anything heavy.

In [ ]:
%%sql duck
SELECT count(*) AS line_items
FROM lakehouse.samples.lineitem

Expected: `60175`.

## 2. Profile every column at once: `SUMMARIZE`

`SUMMARIZE` is a DuckDB feature: one row per column with its type, smallest and largest
value, roughly how many distinct values it has, and how many are missing. It is the fastest
way to get to know a table.

In [ ]:
%%sql duck
SELECT column_name, column_type, min, max, approx_unique, null_percentage
FROM (SUMMARIZE lakehouse.samples.lineitem)

Expected: 16 rows. Things to notice:

- `shipmode` has **7** distinct values; that is what we will compare.
- `commitdate` (the date promised) and `receiptdate` (the date the customer got it) run from
  1992 to 1998.
- `null_percentage` is `0.00` everywhere: no missing values to worry about.

## 3. Define "late"

An item is **late** when it arrived after the promised date: `receiptdate > commitdate`.
`date_diff('day', a, b)` counts the days from `a` to `b`, so a negative number means early.

In [ ]:
%%sql duck
SELECT orderkey, shipmode, commitdate, receiptdate,
       date_diff('day', commitdate, receiptdate) AS days_late
FROM lakehouse.samples.lineitem
LIMIT 5

Expected: 5 rows; some `days_late` are negative (early), some positive (late).

## 4. Look at the whole distribution, as a chart

This time we keep the result in Python (`duck.sql(...).df()` gives a pandas DataFrame), then
draw it. JupyterLab draws **Vega-Lite** charts by itself: you describe the chart (which
column goes on which axis) and JupyterLab renders it. Nothing to install.

In [ ]:
delays = duck.sql("""
    SELECT date_diff('day', commitdate, receiptdate) AS days_late, count(*) AS line_items
    FROM lakehouse.samples.lineitem
    GROUP BY days_late
    ORDER BY days_late
""").df()


def bar_chart(df, x, y, title, x_type="quantitative"):
    """Show a bar chart of DataFrame df in the notebook (Vega-Lite)."""
    spec = {
        "title": title,
        "data": {"values": json.loads(df.to_json(orient="records"))},
        "mark": "bar",
        "encoding": {"x": {"field": x, "type": x_type},
                     "y": {"field": y, "type": "quantitative"}},
        "width": 600, "height": 250,
    }
    display({"application/vnd.vegalite.v5+json": spec}, raw=True)


print(len(delays), "different values of days_late")
bar_chart(delays, "days_late", "line_items", "How many days late (negative = early)")

Expected: `206 different values of days_late`, and a wide hill of bars from about -90 to
+120 days, highest between about -10 and +50. **Most items arrive late**, and by weeks, not days.

## 5. Same question, two engines

Run one query on both engines and compare the answer and the time. `duck.sql(...)` runs in
your workspace; the Trino cursor sends the same SQL to the shared engine.

In [ ]:
sql = """
    SELECT shipmode, count(*) AS line_items
    FROM lakehouse.samples.lineitem
    GROUP BY shipmode
    ORDER BY shipmode
"""
t0 = time.perf_counter()
from_duckdb = duck.sql(sql).fetchall()
t1 = time.perf_counter()
cur = lakehouse.trino_connection().cursor()
cur.execute(sql)
from_trino = cur.fetchall()
t2 = time.perf_counter()

print(f"DuckDB: {t1 - t0:.2f} s   Trino: {t2 - t1:.2f} s")
print("Same answer:", [tuple(r) for r in from_duckdb] == [tuple(r) for r in from_trino])

Expected: `Same answer: True`. Both take well under a second on this small table; which one
is faster varies from run to run. The lesson's step 5 explains when to use which.

Now try `SUMMARIZE` on Trino, to see that the two engines do not speak exactly the same SQL.
(The `try`/`except` shows the error message instead of stopping the notebook.)

In [ ]:
try:
    cur.execute("SUMMARIZE lakehouse.samples.lineitem")
    cur.fetchall()
except Exception as e:
    print("Trino says:", getattr(e, "message", e))

Expected: `Trino says: line 1:1: mismatched input 'SUMMARIZE'. ...`. `SUMMARIZE` exists only in DuckDB.
Most everyday SQL (`SELECT`, `WHERE`, `GROUP BY`, `JOIN`, `FILTER`, `date_diff`) works the
same on both.

## 6. Exercise: late items per shipping mode

Complete the query: for each `shipmode`, count the items, count the late ones, and compute
the percentage that were late. `count(*) FILTER (WHERE ...)` counts only the rows where the
condition is true. Replace the `...` and run.

In [ ]:
by_mode = duck.sql("""
    SELECT shipmode                                              AS ship_mode,
           count(*)                                              AS line_items,
           count(*) FILTER (WHERE receiptdate > commitdate)      AS late_items,
           round(100.0 * count(*) FILTER (WHERE receiptdate > commitdate)
                 / count(*), 1)                                  AS late_pct
    FROM lakehouse.samples.lineitem
    GROUP BY ...
    ORDER BY late_pct DESC
""").df()
by_mode

Expected (7 rows):

| ship_mode | line_items | late_items | late_pct |
|---|---|---|---|
| FOB | 8641 | 5484 | 63.5 |
| MAIL | 8669 | 5467 | 63.1 |
| RAIL | 8566 | 5402 | 63.1 |
| SHIP | 8482 | 5354 | 63.1 |
| AIR | 8491 | 5346 | 63.0 |
| TRUCK | 8710 | 5474 | 62.8 |
| REG AIR | 8616 | 5370 | 62.3 |

Chart it. The y axis starts at **0**, as every bar chart's should: that is what makes the
bars honest.

In [ ]:
bar_chart(by_mode, "ship_mode", "late_pct", "Percent of items delivered late", x_type="nominal")

## 7. Publish the answer (Trino)

The DataFrame `by_mode` lives only in this notebook. To share the answer, save it as a table
in **your schema** with Trino, the same way as in A1. Put the same SELECT (without the
`ORDER BY`) after `AS`; the SQL works unchanged on Trino.

In [ ]:
%%sql trino
CREATE SCHEMA IF NOT EXISTS lakehouse.{{schema}}

In [ ]:
%%sql trino
CREATE OR REPLACE TABLE lakehouse.{{schema}}.a2_late_by_shipmode AS
SELECT ...

In [ ]:
%%sql trino
SELECT * FROM lakehouse.{{schema}}.a2_late_by_shipmode ORDER BY late_pct DESC

Expected: the same 7 rows as in step 6. Trino shows `late_pct` as `63.5000000000000000`:
the same number, kept as an exact decimal (Trino treats `100.0` as a decimal; DuckDB turns the
division into an ordinary floating-point number). Another small difference between engines.

## 8. The answer

Every shipping mode delivers about **63 %** of its items late (62.3 % to 63.5 %). The
differences are far too small to matter: **the data does not support the hunch** that some
modes are much later. Lateness is a problem everywhere, so the team should look elsewhere
(suppliers? order size? season?). Ruling a suspicion out is a real result of exploration.

## 9. Check your work

In [ ]:
!lab-tracks check A2

If `lab-tracks` is not found, run `!python3 checkpoint.py` instead: it is the same check.

**Next:** A3 · your first dbt model.